# AI-Task2: Prompt Quality Tester

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaweriamushtaq41-coder/Smart_District_2/blob/main/AI-Task2/AI_Task2_Prompt_Quality_Tester.ipynb)

**Kya banana hai:** Ek hi kaam (news article ka summary) paanch prompt tareeqon se karwana aur sab jawab CSV mein save karna: zero-shot, few-shot, role-prompted, constrained format, chain-of-thought.

**Concepts:** prompt engineering, zero / few-shot, role prompting, CoT, token count

**Colab mein kaise chalayen:**
1. Upar *Open In Colab* dabao (ya ye file Colab mein upload karo).
2. *Runtime > Run all* dabao.
3. Keys ho to Colab ke left side ke **Secrets** (chaabi ka icon) mein daalo. Bina key ke bhi chalta hai (**demo mode**, nakli jawab).

> Notebook ko share karne se pehle apni keys kabhi output / cell mein mat chhodo.

## 1. Libraries install aur keys (`.env`)

In [1]:
!pip -q install -U python-dotenv openai anthropic google-generativeai groq

In [2]:
import os

# .env file banao. Keys do tareeqon se aa sakti hain:
#  1) Colab "Secrets" (left side ka chaabi wala icon), secret ka naam wahi jo neeche list mein hai, "Notebook access" ON
#  2) apni .env file Colab ke "Files" panel (left side ka folder icon) mein drag-drop kar do, run karne se pehle
NAMES = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "GROQ_API_KEY"]

existing = {}                       # pehle se .env ho (upload ki hui) to uski keys bacha lo
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if "=" in line and not line.startswith("#"):
                key, value = line.strip().split("=", 1)
                existing[key] = value

lines = []
for name in NAMES:
    value = existing.get(name, "")
    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:          # Colab nahi hai, ya ye secret nahi banaya
        pass
    lines.append(f"{name}={value}")

with open(".env", "w") as f:
    f.write("\n".join(lines) + "\n")

from dotenv import load_dotenv
load_dotenv(".env", override=True)

found = [n for n in NAMES if os.getenv(n)]
print("Keys mili:", found if found else "koi nahi (demo mode chalega)")

Keys mili: koi nahi (demo mode chalega)


## 2. Helper: chaaron providers ek jaisa
`ask(provider, prompt, ...)` har provider ke SDK ko bulata hai aur ek hi shakal ka `Reply` deta hai
(text, time, tokens, cost). `demo=True` par nakli jawab milta hai. Ye cell sirf chalana hai, samajhne ke liye parh lo.

In [3]:
import os
import time
from dataclasses import dataclass

from dotenv import load_dotenv

load_dotenv()   # .env file se keys parh leta hai

PROVIDERS = ["openai", "claude", "gemini", "groq"]

# .env mein model ka naam badal sakte ho (model ke naam jaldi purane ho jate hain)
DEFAULT_MODELS = {
    "openai": os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    "claude": os.getenv("ANTHROPIC_MODEL", "claude-3-5-haiku-latest"),
    "gemini": os.getenv("GEMINI_MODEL", "gemini-1.5-flash"),
    "groq": os.getenv("GROQ_MODEL", "llama-3.1-8b-instant"),
}

KEY_NAMES = {
    "openai": "OPENAI_API_KEY",
    "claude": "ANTHROPIC_API_KEY",
    "gemini": "GOOGLE_API_KEY",
    "groq": "GROQ_API_KEY",
}

# (input, output) dollar per 1 million tokens. Ye sirf ANDAZA hai, asli rate provider ki site par dekho.
PRICES = {
    "openai": (0.15, 0.60),
    "claude": (0.80, 4.00),
    "gemini": (0.075, 0.30),
    "groq": (0.05, 0.08),
}


# demo mein har provider ki nakli raftaar (Groq sab se tez, jaisa asal mein hota hai)
DEMO_DELAY = {"openai": 0.12, "claude": 0.18, "gemini": 0.09, "groq": 0.03}


class LLMError(Exception):
    """Provider se jawab na aaye ya key / SDK ka masla ho."""


@dataclass
class Reply:
    provider: str
    model: str
    text: str
    seconds: float
    input_tokens: int
    output_tokens: int
    cost_usd: float
    demo: bool = False

    @property
    def total_tokens(self):
        return self.input_tokens + self.output_tokens


def has_key(provider):
    return bool(os.getenv(KEY_NAMES[provider], "").strip())


def available_providers():
    """Jin providers ki key .env mein mojood hai."""
    return [p for p in PROVIDERS if has_key(p)]


def estimate_cost(provider, input_tokens, output_tokens):
    price_in, price_out = PRICES[provider]
    return (input_tokens * price_in + output_tokens * price_out) / 1_000_000


def rough_tokens(text):
    # angrezi mein taqreeban 4 huroof = 1 token (sirf andaza)
    return max(1, len(text) // 4)


# ---------------- har provider ka apna call ----------------
def _call_openai(model, prompt, system, temperature, max_tokens):
    from openai import OpenAI
    client = OpenAI(api_key=os.getenv(KEY_NAMES["openai"]))
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model=model, messages=messages, temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content, r.usage.prompt_tokens, r.usage.completion_tokens


def _call_groq(model, prompt, system, temperature, max_tokens):
    from groq import Groq
    client = Groq(api_key=os.getenv(KEY_NAMES["groq"]))
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model=model, messages=messages, temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content, r.usage.prompt_tokens, r.usage.completion_tokens


def _call_claude(model, prompt, system, temperature, max_tokens):
    import anthropic
    client = anthropic.Anthropic(api_key=os.getenv(KEY_NAMES["claude"]))
    kwargs = {"model": model, "max_tokens": max_tokens, "temperature": temperature,
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system          # Claude mein system alag parameter hota hai
    r = client.messages.create(**kwargs)
    return r.content[0].text, r.usage.input_tokens, r.usage.output_tokens


def _call_gemini(model, prompt, system, temperature, max_tokens):
    import google.generativeai as genai
    genai.configure(api_key=os.getenv(KEY_NAMES["gemini"]))
    gm = genai.GenerativeModel(model, system_instruction=system)
    r = gm.generate_content(
        prompt, generation_config={"temperature": temperature, "max_output_tokens": max_tokens})
    usage = r.usage_metadata
    return r.text, usage.prompt_token_count, usage.candidates_token_count


CALLERS = {"openai": _call_openai, "claude": _call_claude, "gemini": _call_gemini, "groq": _call_groq}


def _demo_reply(provider, prompt, mock_reply):
    """Nakli jawab: mock_reply (text ya function) ho to wahi, warna ek chhota sa default."""
    if callable(mock_reply):
        text = mock_reply(prompt)
    elif mock_reply is not None:
        text = mock_reply
    else:
        text = (f"[DEMO] Ye {provider} ka nakli jawab hai. Asli jawab ke liye .env mein "
                f"{KEY_NAMES[provider]} likho.")
    return text


def ask(provider, prompt, system=None, temperature=0.7, max_tokens=500,
        demo=False, mock_reply=None, retries=2):
    """Ek provider ko prompt bhejo aur Reply wapas lo (text, time, tokens, cost)."""
    if provider not in CALLERS:
        raise LLMError(f"Unknown provider '{provider}'. Choose from {PROVIDERS}")
    model = DEFAULT_MODELS[provider]
    start = time.perf_counter()

    if demo:
        text = _demo_reply(provider, prompt, mock_reply)
        in_tok = rough_tokens((system or "") + prompt)
        out_tok = rough_tokens(text)
        time.sleep(DEMO_DELAY[provider])         # thora sa waqt lagne ka ehsaas
    else:
        if not has_key(provider):
            raise LLMError(f"{KEY_NAMES[provider]} .env mein nahi mili")
        for attempt in range(retries + 1):
            try:
                text, in_tok, out_tok = CALLERS[provider](model, prompt, system, temperature, max_tokens)
                break
            except ImportError as e:
                raise LLMError(f"{provider} ka SDK install nahi hai: {e}") from e
            except Exception as e:                # SDK ke alag alag errors ek jaisa kar diye
                too_many = "429" in str(e) or "rate" in str(e).lower()
                if too_many and attempt < retries:
                    time.sleep(2 ** attempt)      # 1s, 2s ruk kar dobara
                    continue
                raise LLMError(f"{provider} error: {e}") from e

    seconds = time.perf_counter() - start
    return Reply(provider=provider, model=model, text=text.strip(), seconds=seconds,
                 input_tokens=in_tok, output_tokens=out_tok,
                 cost_usd=estimate_cost(provider, in_tok, out_tok), demo=demo)

## 3. Article aur paanch prompts

In [4]:
import csv
import os
from datetime import datetime

ARTICLE = '''Narowal Tech Park Opens Free Coding Lab for Students

Narowal: A new technology park opened its doors on Monday with a free coding lab for university
students. The lab has 40 computers, high-speed internet and a mentor available every evening.
Officials said more than 300 students from three universities have already registered.

The lab will run weekly workshops on Python, web development and artificial intelligence.
"Our goal is to make sure no student misses a career because of a lack of computers,"
said the park's director. Local companies have promised 25 internship places for the best
participants. The park plans to open a second lab next year if the first six months go well.'''
CSV_FILE = "prompt_results.csv"

In [5]:
FEW_SHOT_EXAMPLE = """Article: The city council approved a new bus route connecting the airport to the old town.
Work will begin in March and finish by December.
Summary: Council approved an airport-to-old-town bus route; work runs March to December.
"""


def build_prompts(article):
    """Paanch variations. System prompt sirf role wale mein alag hai."""
    return {
        "zero-shot": {
            "system": None,
            "prompt": f"Summarise this news article:\n\n{article}",
        },
        "few-shot": {
            "system": None,
            "prompt": (f"Summarise news articles in one sentence, like this example.\n\n"
                       f"{FEW_SHOT_EXAMPLE}\nNow do the same.\nArticle: {article}\nSummary:"),
        },
        "role-prompted": {
            "system": "You are a senior news editor with 20 years of experience. You write clear, neutral summaries.",
            "prompt": f"Summarise this article for the front page.\n\n{article}",
        },
        "constrained-format": {
            "system": None,
            "prompt": (f"Summarise the article below. Rules: exactly 3 bullet points, "
                       f"each under 15 words, no opinions.\n\n\"\"\"\n{article}\n\"\"\""),
        },
        "chain-of-thought": {
            "system": None,
            "prompt": (f"Read the article. First list the key facts step by step. "
                       f"Then write the final summary after the words 'Final summary:'.\n\n{article}"),
        },
    }


# demo mode ke nakli jawab (har tareeqe ka alag andaaz, taake CSV mein farq nazar aaye)
DEMO_REPLIES = {
    "zero-shot": "A new tech park in Narowal opened a free coding lab for students, with 40 computers and evening mentors. "
                 "Over 300 students registered and local companies promised internships.",
    "few-shot": "Narowal Tech Park opened a free 40-computer coding lab; 300+ students registered.",
    "role-prompted": "NAROWAL: A new technology park has opened a free coding lab for university students, "
                     "drawing 300 sign-ups and 25 promised internships from local firms.",
    "constrained-format": "- Free coding lab opened at Narowal Tech Park\n- 40 computers, mentor every evening\n"
                          "- 300 students registered, 25 internships promised",
    "chain-of-thought": "Step 1: A free coding lab opened in Narowal.\nStep 2: It has 40 computers and evening mentors.\n"
                        "Step 3: 300 students registered; 25 internships promised.\n"
                        "Final summary: Narowal's new tech park opened a free coding lab that already has 300 students.",
}


def save_row(row):
    new_file = not os.path.exists(CSV_FILE)
    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if new_file:
            writer.writeheader()
        writer.writerow(row)

## 4. Provider chuno

In [6]:
FORCE_DEMO = False      # True likhoge to hamesha nakli jawab
PROVIDER = None         # "groq", "openai", "claude" ya "gemini". None = jis ki key mili
demo = FORCE_DEMO
provider = PROVIDER or (available_providers() or [None])[0]
if provider is None:
    print("[!] Koi API key nahi mili, DEMO mode chal raha hai.")
    demo, provider = True, "groq"
print("provider =", provider, "| demo =", demo)

[!] Koi API key nahi mili, DEMO mode chal raha hai.
provider = groq | demo = True


## 5. Paanchon prompts chalao aur CSV banao

In [7]:
if os.path.exists(CSV_FILE):
    os.remove(CSV_FILE)                 # naye run ke liye purani file saaf

rows = []
for strategy, parts in build_prompts(ARTICLE).items():
    print(f"Chal raha hai: {strategy} ...")
    try:
        reply = ask(provider, parts["prompt"], system=parts["system"], temperature=0.3,
                    max_tokens=300, demo=demo, mock_reply=DEMO_REPLIES[strategy])
    except LLMError as e:
        print(f"   [!] {e}")
        continue
    row = {
        "time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "provider": reply.provider,
        "strategy": strategy,
        "response": reply.text.replace("\n", " | "),
        "input_tokens": reply.input_tokens,
        "output_tokens": reply.output_tokens,
        "total_tokens": reply.total_tokens,
        "words": len(reply.text.split()),
    }
    save_row(row)
    rows.append(row)

print(f"\n{'Strategy':<20}{'Tokens':>8}{'Words':>8}")
for r in rows:
    print(f"{r['strategy']:<20}{r['total_tokens']:>8}{r['words']:>8}")

Chal raha hai: zero-shot ...
Chal raha hai: few-shot ...
Chal raha hai: role-prompted ...
Chal raha hai: constrained-format ...
Chal raha hai: chain-of-thought ...

Strategy              Tokens   Words
zero-shot                223      28
few-shot                 274      12
role-prompted            243      24
constrained-format       231      22
chain-of-thought         263      42


## 6. CSV dekho
Ye file Colab ke left side ke *Files* mein bhi nazar aati hai (download kar sakti ho).

In [8]:
with open(CSV_FILE, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        print(f"[{row['strategy']}] ({row['total_tokens']} tokens)")
        print("   ", row["response"][:150])

[zero-shot] (223 tokens)
    A new tech park in Narowal opened a free coding lab for students, with 40 computers and evening mentors. Over 300 students registered and local compan
[few-shot] (274 tokens)
    Narowal Tech Park opened a free 40-computer coding lab; 300+ students registered.
[role-prompted] (243 tokens)
    NAROWAL: A new technology park has opened a free coding lab for university students, drawing 300 sign-ups and 25 promised internships from local firms
[constrained-format] (231 tokens)
    - Free coding lab opened at Narowal Tech Park | - 40 computers, mentor every evening | - 300 students registered, 25 internships promised
[chain-of-thought] (263 tokens)
    Step 1: A free coding lab opened in Narowal. | Step 2: It has 40 computers and evening mentors. | Step 3: 300 students registered; 25 internships prom


## Kya seekha
- Prompt ka tareeqa badalne se jawab ki lambai, saaf-pan aur token kharcha badal jata hai.
- Few-shot mein example dene se jawab chhota aur ek jaisa aata hai. Constrained format mein sakht rules milte hain.
- CoT mein jawab lamba (zyada tokens), magar mushkil sawalon mein zyada sahi.
- Sab jawab CSV mein rakhne se muqabla aasan hota hai.